# Justice League Stellar Merger History

## Charlotte Christensen, October 23 2025

This jupyter notebook examins the output Anna Wright's code as modified by Nithun Selva (/home/christenc/Code/python/NithunSelva_startrace/RomZoomSHAnalysisScripts) to identify the stars in those halos.

In [1]:
import pynbody as pb
import numpy as np
import pandas as pd
import glob
import os
import h5py
import tangos

# For importing modules
import importlib.util
import sys
from pathlib import Path

import tangos.examples.mergers as mergers

In [2]:
# Import modules

base_path = '/home/christenc/Code/python/NithunSelva_startrace'

for root, dirs, files in os.walk(base_path):
    if root not in sys.path:
        sys.path.append(root)

In [3]:
simpath = '/data/REPOSITORY/romulus_zooms'
os.environ['TANGOS_SIMULATION_FOLDER'] = simpath
# os.environ['TANGOS_DB_CONNECTION'] = simpath + 'rom25_dwarf_zooms.db'
tangos.core.init_db(simpath + '/rom25_dwarf_zooms.db')
#tangos.all_simulations()

simroot = 'r442'
simname = simroot + '.romulus25.3072g1HsbBH'

In [4]:
# Read in your data
sh_path = '/home/awright/dwarf_stellar_halos/'

with h5py.File(sh_path+'/'+simroot+'/'+simroot+'_allhalostardata.h5','r') as f:
    hostids = f['host_IDs'].asstr()[:] # unique host IDs
    partids = f['particle_IDs'][:] # iords
    pct = f['particle_creation_times'][:] # formation times
    ph = f['particle_hosts'][:] # local host IDs (i.e., host at formation time)
    pp = f['particle_positions'][:] # position at formation time
    tsloc = f['timestep_location'][:] # snapshot where star particle first appears
uIDs = np.unique(hostids)

In [26]:
#halo = tangos.get_halo(simname + "/halo_1")
halo = tangos.get_simulation(simname).timesteps[-1].halos[0]

# first, find all the things that merge into the major progenitor branch
redshift, ratio, progenitor_halos = mergers.get_mergers_of_major_progenitor(halo)
# each item of progenitor_halos is a pair; the first is the major progenitor, the second is the thing merging into it
merging_structures = [x[1] for x in progenitor_halos]

# now here are your properties:

mass_max_tangos_object = [m.calculate('find_progenitor(Mvir, "max")') for m in merging_structures]

max_mass = [m['Mvir'] for m in mass_max_tangos_object]
#mass_at_infall = [m['Mvir'] for m in merging_structures]
time_of_accretion = [m.timestep.time_gyr for m in merging_structures]
time_max_mass = [m.timestep.time_gyr for m in mass_max_tangos_object]
halo_numbers = [m.calculate_for_progenitors('halo_number()') for m in merging_structures]

In [32]:
merging_structures[4]

<Halo 'r442.romulus25.3072g1HsbBH/r442.romulus25.3072g1HsbBH.001248/r442.romulus25.3072g1HsbBH.001248/halo_2' | NDM=298738 Nstar=152037 Ngas=219287>

In [29]:
uIDs

array(['0192_-1', '0192_-2', '0192_14', '0192_2', '0192_24', '0192_274',
       '0192_28', '0192_34', '0288_23', '0288_28', '0288_29', '0384_10',
       '0384_14', '0384_16', '0384_22', '0384_34', '0384_8', '0480_11',
       '0480_26', '0576_-1', '0576_3', '0576_6', '0672_132', '0672_5',
       '1248_2', '3168_16', '4096_1', '4096_2', '4096_3', '4096_32',
       '4096_4', '4096_66', '4096_7', '4096_90'], dtype=object)

In [36]:
#halo = tangos.get_simulation('r442.romulus25.3072g1HsbBH/r442.romulus25.3072g1HsbBH.001248/r442.romulus25.3072g1HsbBH.001248/halo_2')
redshift, ratio, progenitor_halos = mergers.get_mergers_of_major_progenitor(merging_structures[4])
merging_structures = [x[1] for x in progenitor_halos]

In [37]:
merging_structures

[<Halo 'r442.romulus25.3072g1HsbBH/r442.romulus25.3072g1HsbBH.000864/r442.romulus25.3072g1HsbBH.000864/halo_8' | NDM=56868 Nstar=8784 Ngas=48>,
 <Halo 'r442.romulus25.3072g1HsbBH/r442.romulus25.3072g1HsbBH.000672/r442.romulus25.3072g1HsbBH.000672/halo_5' | NDM=116109 Nstar=8785 Ngas=2122>,
 <Halo 'r442.romulus25.3072g1HsbBH/r442.romulus25.3072g1HsbBH.000576/r442.romulus25.3072g1HsbBH.000576/halo_3' | NDM=136243 Nstar=6271 Ngas=143822>,
 <Halo 'r442.romulus25.3072g1HsbBH/r442.romulus25.3072g1HsbBH.000576/r442.romulus25.3072g1HsbBH.000576/halo_6' | NDM=68008 Nstar=5759 Ngas=53289>,
 <Halo 'r442.romulus25.3072g1HsbBH/r442.romulus25.3072g1HsbBH.000480/r442.romulus25.3072g1HsbBH.000480/halo_5' | NDM=119150 Nstar=4963 Ngas=65172>,
 <Halo 'r442.romulus25.3072g1HsbBH/r442.romulus25.3072g1HsbBH.000480/r442.romulus25.3072g1HsbBH.000480/halo_32' | NDM=7124 Nstar=0 Ngas=1>,
 <Halo 'r442.romulus25.3072g1HsbBH/r442.romulus25.3072g1HsbBH.000384/r442.romulus25.3072g1HsbBH.000384/halo_10' | NDM=35891 N

In [39]:
merging_structures[0].keys()

['Mvir',
 'Rvir',
 'GasMass',
 'StarMass',
 'DarkMass',
 'Xc',
 'Yc',
 'Zc',
 'ptcls_in_common',
 'ptcls_in_common',
 'ptcls_in_common',
 'ptcls_in_common']

In [ ]:
#After running .bashrc to set up tangos environment, run the server with:
# tangos serve
# in the terminal


# can supposedly be done from within jupyter notebook with:
#!tangos serve

In [ ]:
# Simulation name and path

# Elena
simname = 'h329'
res = '100'  # The Near Mint runs
#res = 'Mint'  # The Mint
simpath = '/data/Sims/h329.cosmo50PLK.3072g/h329.cosmo50PLK.3072gst5HbwK1BH/'
basename = 'h329.cosmo50PLK.3072gst5HbwK1BH'
ss_dir = 'snapshots_200crit_h329'
sim_base = simpath + ss_dir
ss_z0 = sim_base + basename + '.004096'

outfile_dir = "/home/christenc/Code/Datafiles/stellarhalo_trace_aw/"

# Set the tangos database
# In terminal
# export TANGOS_DB_CONNECTION=/home/christenc/Storage/tangos_db/JL_r200.db
# alternatively:
os.environ['TANGOS_DB_CONNECTION'] = '/home/christenc/Storage/tangos_db/JL_r200.db'